<a href="https://colab.research.google.com/github/Shreya00320/AI_lab/blob/main/Forward_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
FORWARD CHAINING (FOL-FC-ASK) - self-contained

Representation:
  - Variable : lowercase string   -> 'x', 'p'
  - Constant : Capitalised string -> 'Robert', 'T1'
  - Sentence : tuple              -> ('Missile', 'T1') = Missile(T1)
  - Rule     : (list_of_premises, conclusion)
"""


# ---------- small helpers ----------
def is_variable(t):
    return isinstance(t, str) and t[0].islower()


def is_compound(t):
    return isinstance(t, tuple)


def to_str(t):
    if is_compound(t):
        return f"{t[0]}({', '.join(to_str(a) for a in t[1:])})"
    return str(t)


def subst(theta, t):
    if is_variable(t):
        return subst(theta, theta[t]) if t in theta else t
    if is_compound(t):
        return (t[0],) + tuple(subst(theta, a) for a in t[1:])
    return t


def show_theta(theta):
    return "{" + ", ".join(f"{to_str(subst(theta, v))}/{k}"
                           for k, v in theta.items()) + "}"


# ---------- unification (needed for matching) ----------
def occurs_check(var, x, theta):
    if var == x:
        return True
    if is_variable(x) and x in theta:
        return occurs_check(var, theta[x], theta)
    if is_compound(x):
        return any(occurs_check(var, a, theta) for a in x[1:])
    return False


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    if occurs_check(var, x, theta):
        return None
    new_theta = dict(theta)
    new_theta[var] = x
    return new_theta


def unify(x, y, theta):
    if theta is None:
        return None
    if x == y:
        return theta
    if is_variable(x):
        return unify_var(x, y, theta)
    if is_variable(y):
        return unify_var(y, x, theta)
    if is_compound(x) and is_compound(y):
        if x[0] != y[0] or len(x) != len(y):
            return None
        for a, b in zip(x[1:], y[1:]):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


# ---------- forward chaining ----------
def rename(t, suffix):
    """Standardize variables so different rules never clash."""
    if is_variable(t):
        return t + suffix
    if is_compound(t):
        return (t[0],) + tuple(rename(a, suffix) for a in t[1:])
    return t


def match_premises(premises, facts, theta):
    """Yield every substitution that makes ALL premises match known facts."""
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for fact in facts:
        t = unify(first, fact, theta)
        if t is not None:
            yield from match_premises(rest, facts, t)


def fol_fc_ask(facts, rules, query, verbose=True):
    """
    Returns a substitution (dict) if the query is proved, else False.
    NOTE: an empty dict {} means TRUE with nothing to bind,
          so test with 'is not False'.
    """
    kb = set(facts)
    iteration = 0
    while True:                                   # repeat until new is empty
        iteration += 1
        new = set()
        if verbose:
            print(f"\n--- Iteration {iteration} ---")
        for n, (premises, conclusion) in enumerate(rules, 1):
            suffix = f"_{iteration}_{n}"          # standardize variables
            prem = [rename(p, suffix) for p in premises]
            concl = rename(conclusion, suffix)
            for theta in match_premises(prem, list(kb), {}):
                q = subst(theta, concl)           # q' = SUBST(theta, q)
                if q not in kb and q not in new:
                    new.add(q)
                    if verbose:
                        print(f"Rule {n} fires: added {to_str(q)}")
                    phi = unify(q, query, {})     # does it answer the query?
                    if phi is not None:
                        if verbose:
                            print(f"\nQuery {to_str(query)} PROVED")
                        return phi
        if not new:
            if verbose:
                print("\nNo new facts. Query cannot be proved.")
            return False
        kb |= new                                 # add new to KB


# ---------------- DEMO: Robert is criminal ----------------
if __name__ == "__main__":
    facts = [
        ("American", "Robert"),
        ("Enemy", "A", "America"),
        ("Owns", "A", "T1"),
        ("Missile", "T1"),
    ]
    rules = [
        # American(p) ^ Weapon(q) ^ Sells(p,q,r) ^ Hostile(r) => Criminal(p)
        ([("American", "p"), ("Weapon", "q"),
          ("Sells", "p", "q", "r"), ("Hostile", "r")], ("Criminal", "p")),
        # Missile(x) ^ Owns(A,x) => Sells(Robert,x,A)
        ([("Missile", "x"), ("Owns", "A", "x")], ("Sells", "Robert", "x", "A")),
        # Missile(x) => Weapon(x)
        ([("Missile", "x")], ("Weapon", "x")),
        # Enemy(x,America) => Hostile(x)
        ([("Enemy", "x", "America")], ("Hostile", "x")),
    ]

    result = fol_fc_ask(facts, rules, ("Criminal", "Robert"))
    print("Result:", "TRUE " + show_theta(result) if result is not False else "FALSE")

    # Second example: Parent / Grandparent
    print("\n==== Grandparent example ====")
    facts2 = [("Parent", "John", "Mary"), ("Parent", "Mary", "Sam")]
    rules2 = [([("Parent", "x", "y"), ("Parent", "y", "z")],
               ("Grandparent", "x", "z"))]
    result2 = fol_fc_ask(facts2, rules2, ("Grandparent", "John", "Sam"))
    print("Result:", "TRUE " + show_theta(result2) if result2 is not False else "FALSE")


--- Iteration 1 ---
Rule 2 fires: added Sells(Robert, T1, A)
Rule 3 fires: added Weapon(T1)
Rule 4 fires: added Hostile(A)

--- Iteration 2 ---
Rule 1 fires: added Criminal(Robert)

Query Criminal(Robert) PROVED
Result: TRUE {}

==== Grandparent example ====

--- Iteration 1 ---
Rule 1 fires: added Grandparent(John, Sam)

Query Grandparent(John, Sam) PROVED
Result: TRUE {}
